In [ ]:
import os
import re
import pandas as pd

base_dir = r"C:/Users/De"
file_pattern = "AST_urine_H_{year}.xlsx"
years = list(range(2018, 2025))  # 2019–2024 inclusive (change if needed)

# Column names (as in your files)
COL_SEX = "Sex"
COL_AGE = "Age"
COL_MICRO = "Microorganisms"
COL_ABX  = "AST test"
COL_RES  = "AST result"
COL_CNT  = "Overall"

# Output
out_xlsx = os.path.join(base_dir, "MASTER_data.xlsx")

def normalize_city(sheet_name: str) -> str:
    s = str(sheet_name).strip()
    # Remove trailing split suffix like "Astana 1", "Shymkent 2", "Karagandy 1"
    s = re.sub(r"\s+\d+\s*$", "", s).strip()

    # If some years have Karaganda spelling by mistake, force to Karagandy
    if re.match(r"^Karaganda$", s, flags=re.IGNORECASE):
        s = "Karagandy"

    return s

BACT_PATTERNS = [
    (re.compile(r"^E\.?\s*coli$", re.IGNORECASE), "Escherichia coli"),
    (re.compile(r"^Escherichia\s+coli$", re.IGNORECASE), "Escherichia coli"),
    (re.compile(r"^K\.?\s*pneumoniae$", re.IGNORECASE), "Klebsiella pneumoniae"),
    (re.compile(r"^Kleb\s*pn$", re.IGNORECASE), "Klebsiella pneumoniae"),
    (re.compile(r"^Klebsiella\s+pneumoniae$", re.IGNORECASE), "Klebsiella pneumoniae"),
    (re.compile(r"^E\.?\s*faecalis$", re.IGNORECASE), "Enterococcus faecalis"),
    (re.compile(r"^Enterococcus\s+faecalis$", re.IGNORECASE), "Enterococcus faecalis"),
]

def normalize_bacteria(raw_name: str) -> str | None:
    if pd.isna(raw_name):
        return None
    s = str(raw_name).strip()


    for pat, std in BACT_PATTERNS:
        if pat.match(s):
            return std
    return None  # keep only 3 target bacteria

def normalize_result(x: str) -> str | None:
    if pd.isna(x):
        return None
    s = str(x).strip().upper()

    # Exclude not reported
    if s in {"N/R", "NR", "N\\R"}:
        return None

    if s in {"R", "R*"}:
        return "R"
    if s in {"I", "I*"}:
        return "I"
    if s in {"S", "S*"}:
        return "S"

    # Unknown/unexpected -> ignore safely
    return None

AGE_MAP = {
    "0 to 9": "0-19",
    "10 to 19": "0-19",
    "20 to 29": "20-39",
    "30 to 39": "20-39",
    "40 to 49": "40-59",
    "50 to 59": "40-59",
    "60 to 69": "≥60",
    "70 to 90": "≥60",
}

def map_age_group(raw_age_cat: str) -> str | None:
    if pd.isna(raw_age_cat):
        return None
    s = str(raw_age_cat).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return AGE_MAP.get(s)

def clean_sex(raw_sex: str) -> str | None:
    if pd.isna(raw_sex):
        return None
    s = str(raw_sex).strip()
    return s if s else None

def compute_share(df: pd.DataFrame) -> pd.DataFrame:
    # share in percent
    df = df.copy()
    df["share_nonsusceptible"] = (df["n_nonsusceptible"] / df["n_tested"] * 100).where(df["n_tested"] > 0, 0.0)
    return df

# -----------------------------
# LOAD & STACK ALL YEARS/CITIES (raw long with result-level counts)
# -----------------------------
parts = []

for year in years:
    fp = os.path.join(base_dir, file_pattern.format(year=year))
    if not os.path.exists(fp):
        print(f"[SKIP] Not found: {fp}")
        continue

    print(f"[READ] {fp}")
    sheets = pd.read_excel(fp, sheet_name=None, engine="openpyxl")

    for sheet_name, df in sheets.items():
        city = normalize_city(sheet_name)

        required = [COL_SEX, COL_AGE, COL_MICRO, COL_ABX, COL_RES, COL_CNT]
        missing = [c for c in required if c not in df.columns]
        if missing:
            print(f"  [WARN] Sheet '{sheet_name}' missing {missing} -> skipped")
            continue

        sub = df[required].copy()

        # Keep only rows that can contribute to tested/NS:
        # Need antibiotic, result, and count
        sub = sub.dropna(subset=[COL_ABX, COL_RES, COL_CNT])

        # Normalize fields
        sub["year"] = year
        sub["city"] = city
        sub["sex"] = sub[COL_SEX].apply(clean_sex)
        sub["age_group"] = sub[COL_AGE].apply(map_age_group)
        sub["bacteria"] = sub[COL_MICRO].apply(normalize_bacteria)
        sub["antibiotic"] = sub[COL_ABX].astype(str).str.strip()
        sub["result"] = sub[COL_RES].apply(normalize_result)

        # Count
        sub["n"] = pd.to_numeric(sub[COL_CNT], errors="coerce")

        # Keep only valid mapped rows
        sub = sub.dropna(subset=["sex", "age_group", "bacteria", "antibiotic", "result", "n"])
        sub["n"] = sub["n"].astype(int)

        parts.append(sub[["year", "city", "sex", "age_group", "bacteria", "antibiotic", "result", "n"]])

if not parts:
    raise RuntimeError("No data loaded. Check base_dir, file names, and column headers.")

raw = pd.concat(parts, ignore_index=True)

# -----------------------------
# Build long_data (sheet 1): year, city, sex, age_group, bacteria, antibiotic -> tested & NS
# -----------------------------
pivot = (
    raw.pivot_table(
        index=["year", "city", "sex", "age_group", "bacteria", "antibiotic"],
        columns="result",
        values="n",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Ensure S/I/R exist
for col in ["S", "I", "R"]:
    if col not in pivot.columns:
        pivot[col] = 0

pivot["n_tested"] = pivot["S"] + pivot["I"] + pivot["R"]
pivot["n_nonsusceptible"] = pivot["I"] + pivot["R"]

long_data = pivot[["year", "city", "sex", "age_group", "bacteria", "antibiotic", "n_tested", "n_nonsusceptible"]].copy()
long_data = long_data[long_data["n_tested"] > 0].copy()
long_data = compute_share(long_data)

# -----------------------------
# Sheet 2: city_total (aggregate over sex+age)
# -----------------------------
city_total = (
    long_data
    .groupby(["year", "city", "bacteria", "antibiotic"], as_index=False)[["n_tested", "n_nonsusceptible"]]
    .sum()
)
city_total = compute_share(city_total)

# -----------------------------
# Sheet 3: country_total (aggregate over cities + sex + age)
# -----------------------------
country_total = (
    long_data
    .groupby(["year", "bacteria", "antibiotic"], as_index=False)[["n_tested", "n_nonsusceptible"]]
    .sum()
)
country_total = compute_share(country_total)

# -----------------------------
# Sheet 4: age_sex_total (aggregate over cities)
# -----------------------------
age_sex_total = (
    long_data
    .groupby(["year", "sex", "age_group", "bacteria", "antibiotic"], as_index=False)[["n_tested", "n_nonsusceptible"]]
    .sum()
)
age_sex_total = compute_share(age_sex_total)

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    long_data.to_excel(writer, index=False, sheet_name="long_data")
    city_total.to_excel(writer, index=False, sheet_name="city_total")
    country_total.to_excel(writer, index=False, sheet_name="country_total")
    age_sex_total.to_excel(writer, index=False, sheet_name="age_sex_total")

print("\n[DONE] Saved:", out_xlsx)

In [ ]:
import os
import re
import pandas as pd

# -----------------------------
# PATHS
# -----------------------------
base_dir = r"C:/Users/De"
in_master = os.path.join(base_dir, "MASTER_data.xlsx")

out_master = os.path.join(base_dir, "all ABs_2.xlsx")

ABX_MAP_PATTERNS = [
    # Amoxicillin + clavulanic acid variants
    (re.compile(r"^AMOXICILLIN\s*\+\s*CLAVULANIC\s*ACID.*$", re.IGNORECASE), "Amoxicillin/Clavulanic acid"),
    (re.compile(r"^Amox\/K\s*Clav$", re.IGNORECASE), "Amoxicillin/Clavulanic acid"),
    (re.compile(r"^AMOX\/K\s*CLAV$", re.IGNORECASE), "Amoxicillin/Clavulanic acid"),
    (re.compile(r"^Amoxicillin\/K\s*Clavulanate$", re.IGNORECASE), "Amoxicillin/Clavulanic acid"),
    (re.compile(r"^Amoxicillin\s*\+\s*clavulanic\s*acid$", re.IGNORECASE), "Amoxicillin/Clavulanic acid"),

    # Ampicillin + sulbactam variants (DO NOT confuse with Ampicillin alone)
    (re.compile(r"^Amp\/Sulbactam$", re.IGNORECASE), "Ampicillin/Sulbactam"),
    (re.compile(r"^Ampicillin\s*10\s*µg\s*\+\s*sulbactam\s*10\s*µg$", re.IGNORECASE), "Ampicillin/Sulbactam"),

    # Piperacillin + tazobactam variants
    (re.compile(r"^PIPERACILLIN\s*\+\s*TAZOBACTAM\s*30\/6\s*µg$", re.IGNORECASE), "Piperacillin/Tazobactam"),
    (re.compile(r"^PIPERACILLIN\s*\+\s*TAZOBACTAM.*$", re.IGNORECASE), "Piperacillin/Tazobactam"),
    (re.compile(r"^Pip\/Tazo$", re.IGNORECASE), "Piperacillin/Tazobactam"),

    # TMP/SMX variants
    (re.compile(
        r"^TRIMETHOPRIM\s*1[,\.]25\s*µg\s*\+\s*SULFAMETHOXAZOLE\s*23[,\.]75\s*µg$",
        re.IGNORECASE
    ), "Trimethoprim/Sulfamethoxazole"),
    (re.compile(r"^Trimeth\/Sulfamethoxazole$", re.IGNORECASE), "Trimethoprim/Sulfamethoxazole"),
    (re.compile(
        r"^Trimetoprim\s*1[,\.]25\s*µg\s*\+\s*Sulfamethoxazole\s*23[,\.]75\s*µg$",
        re.IGNORECASE
    ), "Trimethoprim/Sulfamethoxazole"),

    # Chloramphenicol variants
    (re.compile(r"^Chloramphenicol\(Левомицетин\)$", re.IGNORECASE), "Chloramphenicol"),
    (re.compile(r"^Chloramphenicol$", re.IGNORECASE), "Chloramphenicol"),
]

def normalize_other_antibiotics(a: str) -> str | None:
    if pd.isna(a):
        return None
    s = str(a).strip()
    s = re.sub(r"\s+", " ", s)

    for pat, std in ABX_MAP_PATTERNS:
        if pat.match(s):
            return std
    return s  # unchanged if not matched

def recompute_share(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["share_nonsusceptible"] = (df["n_nonsusceptible"] / df["n_tested"] * 100).where(df["n_tested"] > 0, 0.0)
    return df

def normalize_and_aggregate(df: pd.DataFrame, group_cols: list) -> pd.DataFrame:
    """
    df must contain columns:
      group_cols + ['n_tested','n_nonsusceptible'] and specifically 'antibiotic'
    """
    df = df.copy()
    if "antibiotic" not in df.columns:
        raise ValueError("Expected column 'antibiotic' not found.")

    df["antibiotic"] = df["antibiotic"].apply(normalize_other_antibiotics)

    # Re-aggregate because we merged names
    out = (
        df.groupby(group_cols, as_index=False)[["n_tested", "n_nonsusceptible"]]
          .sum()
    )
    out = out[out["n_tested"] > 0].copy()
    out = recompute_share(out)

    # final column order
    out = out[group_cols + ["n_tested", "n_nonsusceptible", "share_nonsusceptible"]].copy()
    return out

# -----------------------------
# LOAD + PROCESS 4 SHEETS
# -----------------------------
print("[READ]", in_master)
sheets = pd.read_excel(in_master, sheet_name=None, engine="openpyxl")
need = {"long_data", "city_total", "country_total", "age_sex_total"}
missing = need - set(sheets.keys())
if missing:
    raise ValueError(f"Missing sheets in input master: {sorted(missing)}")

long_df = sheets["long_data"]
city_df = sheets["city_total"]
country_df = sheets["country_total"]
age_sex_df = sheets["age_sex_total"]

long_cols = ["year", "city", "sex", "age_group", "bacteria", "antibiotic"]
city_cols = ["year", "city", "bacteria", "antibiotic"]
country_cols = ["year", "bacteria", "antibiotic"]
age_sex_cols = ["year", "sex", "age_group", "bacteria", "antibiotic"]

long2 = normalize_and_aggregate(long_df, long_cols)
city2 = normalize_and_aggregate(city_df, city_cols)
country2 = normalize_and_aggregate(country_df, country_cols)
age_sex2 = normalize_and_aggregate(age_sex_df, age_sex_cols)

# -----------------------------
# SAVE
# -----------------------------
with pd.ExcelWriter(out_master, engine="openpyxl") as writer:
    long2.to_excel(writer, index=False, sheet_name="long_data")
    city2.to_excel(writer, index=False, sheet_name="city_total")
    country2.to_excel(writer, index=False, sheet_name="country_total")
    age_sex2.to_excel(writer, index=False, sheet_name="age_sex_total")

print("[DONE] Saved:", out_master)

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = r"C:/Users/De"
IN_FILE = os.path.join(BASE_DIR, "all ABs_2.xlsx")

SHEETS = ["E coli", "K pneumoniae", "E faecalis"]

# Selected antibiotics (as you specified)
ABX_BY_BUG = {
    "Escherichia coli": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Klebsiella pneumoniae": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Enterococcus faecalis": [
        "Ampicillin", "Gentamicin", "Linezolid", "Nitrofurantoin", "Vancomycin"
    ],
}

# ---- helpers
def _clean_antibiotic_label(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("Piperacillin/Tazobactam", "Piperacillin/Tazobactam")
    # normalize common variants (safe; adjust if needed)
    s_low = s.lower()
    if s_low in {"pip/tazo", "piperacillin + tazobactam", "piperacillin + tazobactam 30/6 µg"}:
        return "Piperacillin/Tazobactam"
    if s_low in {"amoxicillin/k clavulanate", "amox/k clav", "amoxicillin + clavulanic acid"}:
        return "Amoxicillin/Clavulanic acid"
    return s

def _clean_bacteria_label(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    # minimal normalization (in case)
    if s.lower() in {"e coli", "escherichia coli"}:
        return "Escherichia coli"
    if s.lower() in {"k pneumoniae", "klebsiella pneumoniae"}:
        return "Klebsiella pneumoniae"
    if s.lower() in {"e faecalis", "enterococcus faecalis"}:
        return "Enterococcus faecalis"
    return s

# Build one global color map so antibiotics keep same color across plots
# (Matplotlib will pick distinct colors automatically from tab20)
ALL_ABX = sorted({abx for lst in ABX_BY_BUG.values() for abx in lst})
cmap = plt.get_cmap("tab20")
COLOR_MAP = {abx: cmap(i % 20) for i, abx in enumerate(ALL_ABX)}

def plot_one_bacteria(df: pd.DataFrame, bug: str):
    # Expect columns (typical): year, bacteria, antibiotic, n_tested, n_nonsusceptible, share_nonsusceptible
    # Tolerate slight column name variations:
    col_year = "year"
    col_bug = "bacteria"
    col_abx = "antibiotic"
    col_share = "share_nonsusceptible"

    for c in [col_year, col_bug, col_abx, col_share]:
        if c not in df.columns:
            raise ValueError(f"Missing required column '{c}' in the sheet. Found: {list(df.columns)}")

    tmp = df.copy()
    tmp[col_bug] = tmp[col_bug].apply(_clean_bacteria_label)
    tmp[col_abx] = tmp[col_abx].apply(_clean_antibiotic_label)

    tmp = tmp[tmp[col_bug] == bug].copy()
    tmp = tmp[tmp[col_abx].isin(ABX_BY_BUG[bug])].copy()

    # Make sure year is integer
    tmp[col_year] = pd.to_numeric(tmp[col_year], errors="coerce")
    tmp = tmp.dropna(subset=[col_year, col_share])
    tmp[col_year] = tmp[col_year].astype(int)

    plt.figure(figsize=(10, 5))
    ax = plt.gca()

    for abx in ABX_BY_BUG[bug]:
        d = tmp[tmp[col_abx] == abx].sort_values(col_year)
        if d.empty:
            continue

        x = d[col_year].to_list()
        y = d[col_share].to_list()

        ax.plot(x, y, marker="o", linewidth=2, label=abx, color=COLOR_MAP.get(abx))

        # annotate each point with percent
        for xi, yi in zip(x, y):
            if pd.isna(yi):
                continue
            ax.text(xi, yi, f"{yi:.1f}%", ha="center", va="bottom", fontsize=9)

    ax.set_title(bug, fontsize=14)
    ax.set_xlabel("Year")
    ax.set_ylabel("Nonsusceptible share (%)")
    ax.set_xticks(sorted(tmp[col_year].unique().tolist()))
    ax.grid(True, alpha=0.3)

    # legend to the right
    ax.legend(title="Antibiotic", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

    plt.tight_layout()
    plt.show()

# ---- Read sheets and plot
all_sheets = pd.read_excel(IN_FILE, sheet_name=None, engine="openpyxl")

# If your sheets contain ONLY that bacteria (most likely), we still identify bug name safely:
SHEET_TO_BUG = {
    "E coli": "Escherichia coli",
    "K pneumoniae": "Klebsiella pneumoniae",
    "E faecalis": "Enterococcus faecalis",
}

for sh in SHEETS:
    if sh not in all_sheets:
        raise ValueError(f"Sheet '{sh}' not found in {IN_FILE}. Found sheets: {list(all_sheets.keys())}")
    plot_one_bacteria(all_sheets[sh], SHEET_TO_BUG[sh])
# 2) AAPC (95% CI) for the selected bug–antibiotic pairs
#    - Uses Poisson regression on NS counts with offset log(n_tested):
#        log(NS_count) = a + b*year + log(n_tested)
#      => annual percent change (APC) = (exp(b) - 1)*100
#    - 95% CI from b ± 1.96*SE
#    - p-value from the model
#    - Filters: n_tested >= 20 PER YEAR (within each bug–abx series)
#    - Saves to Excel in C:/Users/De

import os
import numpy as np
import pandas as pd

import statsmodels.api as sm

BASE_DIR = r"C:/Users/De"
IN_FILE = os.path.join(BASE_DIR, "all ABs_2.xlsx")
OUT_AAPC = os.path.join(BASE_DIR, "AAPC_selected_pairs_2019-2024.xlsx")

SHEETS = ["E coli", "K pneumoniae", "E faecalis"]
SHEET_TO_BUG = {
    "E coli": "Escherichia coli",
    "K pneumoniae": "Klebsiella pneumoniae",
    "E faecalis": "Enterococcus faecalis",
}

ABX_BY_BUG = {
    "Escherichia coli": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Klebsiella pneumoniae": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Enterococcus faecalis": [
        "Ampicillin", "Gentamicin", "Linezolid", "Nitrofurantoin", "Vancomycin"
    ],
}

def _clean_antibiotic_label(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    s_low = s.lower()
    if s_low in {"pip/tazo", "piperacillin + tazobactam", "piperacillin + tazobactam 30/6 µg"}:
        return "Piperacillin/Tazobactam"
    return s

def _clean_bacteria_label(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in {"e coli", "escherichia coli"}:
        return "Escherichia coli"
    if s.lower() in {"k pneumoniae", "klebsiella pneumoniae"}:
        return "Klebsiella pneumoniae"
    if s.lower() in {"e faecalis", "enterococcus faecalis"}:
        return "Enterococcus faecalis"
    return s

def compute_apc_poisson(d: pd.DataFrame) -> dict:
    """
    d: dataframe for ONE bacteria+antibiotic across years with cols:
       year, n_tested, n_nonsusceptible
    Returns APC (as AAPC for this single-segment model), CI, p-value, n_years, year_range.
    """
    dd = d.copy()
    dd["year"] = pd.to_numeric(dd["year"], errors="coerce")
    dd["n_tested"] = pd.to_numeric(dd["n_tested"], errors="coerce")
    dd["n_nonsusceptible"] = pd.to_numeric(dd["n_nonsusceptible"], errors="coerce")
    dd = dd.dropna(subset=["year", "n_tested", "n_nonsusceptible"]).copy()
    dd["year"] = dd["year"].astype(int)

    # Filter per-year minimum tests
    dd = dd[dd["n_tested"] >= 20].copy()

    if dd.shape[0] < 3:
        return {"status": "too_few_points"}  # not enough to fit trend reliably

    # Offset = log(n_tested)
    # NS count can be 0; add a small continuity correction to avoid log-likelihood issues
    # (keeps series, avoids dropping years)
    dd["ns_cc"] = dd["n_nonsusceptible"].astype(float)
    dd.loc[dd["ns_cc"] <= 0, "ns_cc"] = 0.5

    X = sm.add_constant(dd["year"].astype(float))
    y = dd["ns_cc"].astype(float)
    offset = np.log(dd["n_tested"].astype(float))

    model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset)
    res = model.fit()

    b = res.params["year"]
    se = res.bse["year"]
    p = res.pvalues["year"]

    apc = (np.exp(b) - 1.0) * 100.0
    ci_low = (np.exp(b - 1.96 * se) - 1.0) * 100.0
    ci_high = (np.exp(b + 1.96 * se) - 1.0) * 100.0

    return {
        "status": "ok",
        "n_years_used": int(dd.shape[0]),
        "year_min": int(dd["year"].min()),
        "year_max": int(dd["year"].max()),
        "AAPC_percent": float(apc),
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "p_value": float(p),
    }

# ---- Load all sheets
sheets = pd.read_excel(IN_FILE, sheet_name=None, engine="openpyxl")

# ---- Collect AAPC results
rows = []
for sh in SHEETS:
    if sh not in sheets:
        raise ValueError(f"Sheet '{sh}' not found in {IN_FILE}. Found: {list(sheets.keys())}")

    bug = SHEET_TO_BUG[sh]
    df = sheets[sh].copy()

    # Clean labels
    df["bacteria"] = df["bacteria"].apply(_clean_bacteria_label) if "bacteria" in df.columns else bug
    df["antibiotic"] = df["antibiotic"].apply(_clean_antibiotic_label)

    # Ensure required columns exist
    for col in ["year", "antibiotic", "n_tested", "n_nonsusceptible"]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in sheet '{sh}'. Found: {list(df.columns)}")

    # Force bacteria if sheet is single-bug
    df["bacteria"] = bug

    for abx in ABX_BY_BUG[bug]:
        d = df[(df["antibiotic"] == abx)].copy()
        if d.empty:
            rows.append({
                "bacteria": bug, "antibiotic": abx, "status": "no_data"
            })
            continue

        out = compute_apc_poisson(d)
        rows.append({
            "bacteria": bug,
            "antibiotic": abx,
            **out
        })

res_df = pd.DataFrame(rows)

# Nicely format
order_cols = [
    "bacteria", "antibiotic", "status",
    "n_years_used", "year_min", "year_max",
    "AAPC_percent", "CI95_low", "CI95_high", "p_value"
]
for c in order_cols:
    if c not in res_df.columns:
        res_df[c] = np.nan
res_df = res_df[order_cols].sort_values(["bacteria", "antibiotic"]).reset_index(drop=True)

# Save to Excel
with pd.ExcelWriter(OUT_AAPC, engine="openpyxl") as writer:
    res_df.to_excel(writer, index=False, sheet_name="AAPC_results")

print("[DONE] Saved AAPC results to:", OUT_AAPC)
print("Rows:", len(res_df))

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BASE_DIR = r"C:/Users/De"
IN_FILE = os.path.join(BASE_DIR, "all ABs_2.xlsx")
OUT_DIR = os.path.join(BASE_DIR, "plots_png_country_total_no_labels")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Target bacteria & antibiotics ----------------
ABX_BY_BUG = {
    "Escherichia coli": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Klebsiella pneumoniae": [
        "Ampicillin", "Ceftriaxone", "Meropenem", "Ciprofloxacin",
        "Piperacillin/Tazobactam", "Amikacin", "Nitrofurantoin"
    ],
    "Enterococcus faecalis": [
        "Ampicillin", "Gentamicin", "Linezolid", "Nitrofurantoin", "Vancomycin"
    ],
}

ABX_SHORT = {
    "ampicillin": "AMP",
    "gentamicin": "GEN",
    "linezolid": "LZD",
    "nitrofurantoin": "NIT",
    "vancomycin": "VAN",
    "ceftriaxone": "CRO",
    "meropenem": "MER",
    "ciprofloxacin": "CIP",
    "piperacillin/tazobactam": "PTZ",
    "amikacin": "AMK",
}

def short_abx(name: str) -> str:
    return ABX_SHORT.get(str(name).strip().lower(), str(name).strip())

def clean_antibiotic(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    s_low = s.lower()

    # normalize common variants
    if s_low in {"pip/tazo", "piperacillin + tazobactam", "piperacillin+tazobactam"}:
        return "Piperacillin/Tazobactam"
    if s_low in {"piperacillin/tazobactam"}:
        return "Piperacillin/Tazobactam"

    # keep canonical casing if matches our target list
    # (otherwise return original cleaned)
    return s

def make_color_map(all_abx: list[str]) -> dict:
    # Stable per-antibiotic color map across all bacteria plots
    cmap = plt.get_cmap("tab10")
    return {abx: cmap(i % 10) for i, abx in enumerate(all_abx)}

ALL_ABX = sorted({a for lst in ABX_BY_BUG.values() for a in lst})
COLOR_MAP = make_color_map(ALL_ABX)

# ---------------- Column detection helpers ----------------
def detect_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def standardize_bug_name(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()

    # common variants
    if s in {"e. coli", "e coli", "escherichia coli"}:
        return "Escherichia coli"
    if s in {"k. pneumoniae", "k pneumoniae", "klebsiella pneumoniae"}:
        return "Klebsiella pneumoniae"
    if s in {"e. faecalis", "e faecalis", "enterococcus faecalis"}:
        return "Enterococcus faecalis"

    return str(x).strip()

def plot_bacteria(df: pd.DataFrame, bug: str, out_png: str):
    # Detect required columns with some flexibility
    col_year = detect_column(df, ["year", "Year", "YYYY"])
    col_abx  = detect_column(df, ["antibiotic", "Antibiotic", "abx", "ABX", "drug", "Drug"])
    col_bug  = detect_column(df, ["bacteria", "Bacteria", "pathogen", "Pathogen", "organism", "Organism"])
    col_share = detect_column(df, [
        "share_nonsusceptible", "Share_nonsusceptible", "nonsusceptible_share",
        "NS_share", "ns_share", "share_ns", "share_NS", "share", "Share"
    ])
    col_n = detect_column(df, ["n_tested", "N_tested", "tested", "n", "N", "tests", "n_tests"])

    missing = [k for k, v in {
        "year": col_year,
        "antibiotic": col_abx,
        "bacteria": col_bug,
        "share_nonsusceptible": col_share,
        "n_tested": col_n
    }.items() if v is None]

    if missing:
        raise ValueError(
            f"[{bug}] Missing columns: {missing}. "
            f"Found columns: {list(df.columns)}"
        )

    tmp = df.copy()

    tmp[col_abx] = tmp[col_abx].apply(clean_antibiotic)
    tmp[col_bug] = tmp[col_bug].apply(standardize_bug_name)

    tmp[col_year] = pd.to_numeric(tmp[col_year], errors="coerce")
    tmp[col_share] = pd.to_numeric(tmp[col_share], errors="coerce")
    tmp[col_n] = pd.to_numeric(tmp[col_n], errors="coerce")

    # Keep only this bug + target antibiotics + n_tested >= 20
    keep_abx = ABX_BY_BUG[bug]
    tmp = tmp[tmp[col_bug] == bug].copy()
    tmp = tmp[tmp[col_abx].isin(keep_abx)].copy()
    tmp = tmp.dropna(subset=[col_year, col_share, col_n]).copy()
    tmp = tmp[tmp[col_n] >= 20].copy()

    
    tmp[col_year] = tmp[col_year].astype(int)

    years = sorted(tmp[col_year].unique().tolist())
    if not years:
        raise ValueError(f"[{bug}] No years available for display after filtering.")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_title(bug, fontsize=16)
    ax.set_xlabel("Year")
    ax.set_ylabel("Nonsusceptible share (%)")
    ax.set_xticks(years)
    ax.grid(True, alpha=0.3)

    present_abx = []

    # Plot each antibiotic (no value labels/annotations)
    for abx in keep_abx:
        d = tmp[tmp[col_abx] == abx].sort_values(col_year)
        if d.empty:
            continue

        x = d[col_year].tolist()
        y = d[col_share].tolist()

        ax.plot(
            x, y,
            marker="o",
            linewidth=2,
            color=COLOR_MAP[abx],      # consistent across plots
            label=short_abx(abx)
        )

        present_abx.append(abx)

    # Legend bottom INSIDE figure
    handles = [mpatches.Patch(color=COLOR_MAP[abx], label=short_abx(abx)) for abx in present_abx]
    fig.subplots_adjust(bottom=0.22)
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=min(len(handles), 8),
        frameon=False,
        handlelength=1.0,
        handletextpad=0.2,
        columnspacing=1.0
    )

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# ---------------- Run: read ONLY first sheet ----------------
# Read all sheet names, then load the first one (assumed "country total")
xls = pd.ExcelFile(IN_FILE, engine="openpyxl")
first_sheet_name = xls.sheet_names[0]
df0 = pd.read_excel(IN_FILE, sheet_name=first_sheet_name, engine="openpyxl")

print(f"[INFO] Using first sheet: '{first_sheet_name}'")

for bug in ABX_BY_BUG.keys():
    out_png = os.path.join(OUT_DIR, f"trend_{bug.replace(' ', '_')}_NS_share_country_total_no_labels.png")
    plot_bacteria(df0, bug, out_png)

print("[DONE] PNG saved to:", OUT_DIR)

# 2) REPLACEMENT BLOCK: custom multi-panel layout
#    Top row: Escherichia coli | Klebsiella pneumoniae (shared legend for Gram-neg antibiotics)
#    Bottom row: Enterococcus faecalis (its own legend)
#
# Saves one PNG to OUT_DIR

# ---- Define antibiotic groups for legends
GN_BUGS = ["Escherichia coli", "Klebsiella pneumoniae"]
GP_BUG = "Enterococcus faecalis"

GN_ABX = ABX_BY_BUG["Escherichia coli"]  # same list as for Klebsiella pneumoniae
GP_ABX = ABX_BY_BUG["Enterococcus faecalis"]

# ---- Figure with GridSpec: 2 columns on top, bottom spans both columns
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[1, 1.05], hspace=0.35, wspace=0.18)

ax_ecoli = fig.add_subplot(gs[0, 0])
ax_kpneu = fig.add_subplot(gs[0, 1])
ax_efaec = fig.add_subplot(gs[1, :])  # span both columns

# ---- Plot panels
_ = plot_bacteria_on_ax(ax_ecoli, df0, "Escherichia coli")
_ = plot_bacteria_on_ax(ax_kpneu, df0, "Klebsiella pneumoniae")
_ = plot_bacteria_on_ax(ax_efaec, df0, "Enterococcus faecalis")

# ---- Legends: top shared (Gram-neg), bottom separate (Enterococcus)
handles_gn = [mpatches.Patch(color=COLOR_MAP[abx], label=short_abx(abx)) for abx in GN_ABX]
handles_gp = [mpatches.Patch(color=COLOR_MAP[abx], label=short_abx(abx)) for abx in GP_ABX]

# Position legends just below the corresponding row
ax_ecoli.legend(
    handles=handles_gn,
    loc="upper center",
    bbox_to_anchor=(1.05, -0.22),  # centered under the two top axes
    ncol=min(len(handles_gn), 7),
    frameon=False,
    handlelength=1.0,
    handletextpad=0.3,
    columnspacing=1.0
)

ax_efaec.legend(
    handles=handles_gp,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),   # centered under bottom axis
    ncol=min(len(handles_gp), 5),
    frameon=False,
    handlelength=1.0,
    handletextpad=0.3,
    columnspacing=1.0
)

out_panel_png = os.path.join(OUT_DIR, "NStrends_panel.png")
fig.savefig(out_panel_png, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print("[DONE] Custom panel PNG saved to:", out_panel_png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# -----------------------------
# SETTINGS
# -----------------------------
FILE_PATH = r"C:\Users\De\all ABs_2.xlsx"
OUT_PNG = r"C:\Users\De\heatmap"
SHEET_NAME = "country total"


YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

BAC_ORDER = ["E.faecalis", "E. coli", "K.pneumoniae"]

NO_DATA_GREY = "#eeeeee"
LOW_N = 20  # only for ★ marking

# -----------------------------
# ANTIBIOTIC RECODING (AS PROVIDED)
# -----------------------------
ABX_CODE = {
    # Penicillins / BL-BLI
    "Ampicillin": "AMP",
    "Amoxicillin": "AMX",
    "Amoxicillin/Clavulanic acid": "AMC",
    "Ampicillin/Sulbactam": "SAM",
    "Piperacillin": "PIP",
    "Piperacillin/Tazobactam": "PTZ",
    # Cephalosporins
    "Cefazolin": "CFZ",
    "Cefuroxime": "CXM",
    "Cefotaxime": "CTX",
    "Ceftriaxone": "CRO",
    "Ceftazidime": "CAZ",
    "Cefepime": "FEP",
    "Cefoxitin": "FOX",
    "Cefoperazone": "CFP",
    "Ceftazidime/Avibactam": "CZA",
    "Ceftazidime/K Clavulanate": "CZC",
    # Monobactam
    "Aztreonam": "ATM",
    # Carbapenems
    "Ertapenem": "ETP",
    "Imipenem": "IPM",
    "Meropenem": "MER",
    # Fluoroquinolones
    "Ciprofloxacin": "CIP",
    "Levofloxacin": "LVX",
    "Moxifloxacin": "MXF",
    # Aminoglycosides
    "Gentamicin": "GEN",
    "Amikacin": "AMK",
    "Tobramycin": "TOB",
    # UTI / Other
    "Nitrofurantoin": "NIT",
    "Fosfomycin": "FOS",
    "Trimethoprim/Sulfamethoxazole": "SXT",
    "Colistin": "CST",
    "Tetracycline": "TET",
    "Tigecycline": "TGC",
    # Enterococcus-specific
    "Vancomycin": "VAN",
    "Linezolid": "LZD",
    "Daptomycin": "DAP",
}

AB_ORDER = list(dict.fromkeys(ABX_CODE.values()))

# -----------------------------
# READ
# -----------------------------
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

expected_cols = [
    "year",
    "bacteria",
    "antibiotic",
    "n_tested",
    "n_nonsusceptible",
    "share_nonsusceptible",
]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Columns in file: {list(df.columns)}"
    )

# -----------------------------
# CLEAN (MINIMAL, STRICT)
# -----------------------------
df = df.copy()

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df[df["year"].isin(YEARS)]

df["bacteria"] = df["bacteria"].astype(str).str.strip()
df = df[df["bacteria"].isin(BAC_ORDER)]

# recode antibiotics (long -> short)
df["antibiotic"] = df["antibiotic"].astype(str).str.strip()
df["antibiotic_code"] = df["antibiotic"].map(ABX_CODE)

# DEBUG: show what matched
print("Mapped antibiotic codes:", sorted(df["antibiotic_code"].dropna().unique()))

df = df[df["antibiotic_code"].isin(AB_ORDER)]

df["n_tested"] = pd.to_numeric(df["n_tested"], errors="coerce").fillna(0).astype(int)
df["share_nonsusceptible"] = pd.to_numeric(df["share_nonsusceptible"], errors="coerce")

# -----------------------------
# BUILD MATRICES
# -----------------------------
mats = {}

for bac in BAC_ORDER:
    d = df[df["bacteria"] == bac]

    share = (
        d.pivot_table(
            index="year",
            columns="antibiotic_code",
            values="share_nonsusceptible",
            aggfunc="mean"
        )
        .reindex(index=YEARS, columns=AB_ORDER)
    )

    n = (
        d.pivot_table(
            index="year",
            columns="antibiotic_code",
            values="n_tested",
            aggfunc="sum"
        )
        .reindex(index=YEARS, columns=AB_ORDER)
        .fillna(0)
        .astype(int)
    )

    # ONLY mask true no-data
    share_show = share.mask(n == 0)

    mats[bac] = {"share": share_show, "n": n}

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(18.5, 6.2),
    sharex=True,
    gridspec_kw={"hspace": 0}
)

cmap = plt.cm.Reds.copy()
cmap.set_bad(NO_DATA_GREY)
norm = Normalize(vmin=0, vmax=100)

mappable = None

for i, bac in enumerate(BAC_ORDER):
    ax = axes[i]

    data = mats[bac]["share"].values
    nmat = mats[bac]["n"].values

    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        norm=norm,
        interpolation="none"
    )

    if mappable is None:
        mappable = im

    ax.set_yticks(range(len(YEARS)))
    ax.set_yticklabels(YEARS)

    ax.text(
        -0.04, 0.3, bac,
        transform=ax.transAxes,
        ha="right",
        va="center",
        fontsize=12,
    )
 
    # ★ where 0 < n_tested < 20
    for r in range(nmat.shape[0]):
        for c in range(nmat.shape[1]):
            if 0 < nmat[r, c] < LOW_N:
                ax.text(
                    c - 0.45,
                    r - 0.35,
                    "*",
                    fontsize=9,
                    ha="left",
                    va="top"
                )

    if i < 2:
        ax.axhline(len(YEARS) - 0.5, color="black", lw=1)

    for spine in ax.spines.values():
        spine.set_linewidth(1)

axes[-1].set_xticks(range(len(AB_ORDER)))
axes[-1].set_xticklabels(AB_ORDER)
axes[-1].tick_params(axis="x", length=0)

for ax in axes[:-1]:
    ax.tick_params(axis="x", bottom=False, labelbottom=False)

cbar = fig.colorbar(mappable, ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Nonsusceptible share (%)")

plt.savefig(OUT_PNG, dpi=300)
plt.show()

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency

BASE_DIR = Path(r"C:/Users/De")
IN_PATH = BASE_DIR / "all ABs_2.xlsx"


# Column names (exactly as you wrote / as in your Excel)
COL_CITY = "region"
COL_TESTED = "n_tested"
COL_NS = "n_nonsusceptible"

# Load all sheets
xl = pd.ExcelFile(IN_PATH)
sheet_names = xl.sheet_names

results = []

for sh in sheet_names:
    df = pd.read_excel(IN_PATH, sheet_name=sh)

    # strip possible hidden spaces in headers
    df.columns = [str(c).strip() for c in df.columns]

    # validate required columns
    need = [COL_CITY, COL_TESTED, COL_NS]
    miss = [c for c in need if c not in df.columns]
    if miss:
        print(f"[SKIP] Sheet '{sh}': missing columns {miss}")
        continue

    d = df[[COL_CITY, COL_TESTED, COL_NS]].copy()
    d[COL_CITY] = d[COL_CITY].astype(str).str.strip()

    d[COL_TESTED] = pd.to_numeric(d[COL_TESTED], errors="coerce").fillna(0)
    d[COL_NS] = pd.to_numeric(d[COL_NS], errors="coerce").fillna(0)

    # keep only cities with tested > 0
    d = d[d[COL_TESTED] > 0].copy()

    if d.empty:
        results.append({
            "sheet": sh,
            "n_cities_used": 0,
            "chi2": np.nan,
            "df": np.nan,
            "p_value": np.nan,
            "cramers_v": np.nan,
            "note": "All n_tested are 0 (no data after filtering)."
        })
        continue

    # compute susceptible counts
    d["n_susceptible"] = d[COL_TESTED] - d[COL_NS]

    # safety: if any negative (data issue) -> stop for that sheet
    if (d["n_susceptible"] < 0).any():
        bad = d.loc[d["n_susceptible"] < 0, [COL_CITY, COL_TESTED, COL_NS]]
        raise ValueError(f"Sheet '{sh}': found n_nonsusceptible > n_tested in:\n{bad}")

    # Build 2 x K contingency table: [NS; S] x cities
    cities = d[COL_CITY].tolist()
    table = np.vstack([
        d[COL_NS].to_numpy(dtype=float),
        d["n_susceptible"].to_numpy(dtype=float)
    ])

    # Pearson chi-square
    chi2, p, dof, expected = chi2_contingency(table, correction=False)

    # Cramér’s V for RxC: V = sqrt(chi2 / (n * (min(r-1, c-1))))
    n = table.sum()
    r, c = table.shape
    denom = n * (min(r - 1, c - 1))
    cramers_v = np.sqrt(chi2 / denom) if denom > 0 else np.nan

    results.append({
        "sheet": sh,
        "n_cities_used": len(cities),
        "chi2": chi2,
        "df": dof,
        "p_value": p,
        "cramers_v": cramers_v,
        "note": ""
    })

res = pd.DataFrame(results).sort_values("sheet").reset_index(drop=True)

print("Input:", IN_PATH)
print("\nResults:")
print(res)

# Save results into a new sheet "stats" (keeps original sheets)
OUT_PATH = IN_PATH  # overwrite same file by adding/refreshing a sheet
with pd.ExcelWriter(OUT_PATH, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    res.to_excel(writer, sheet_name="stats", index=False)

print(f"\nSaved sheet 'stats' into: {OUT_PATH}")

In [ ]:
# M_logistic_regression

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter

# -----------------------------
# PATHS
# -----------------------------
xlsx_in = Path(r"C:\Users\De\data_for_regression.xlsx")
if not xlsx_in.exists():
    xlsx_in = Path("/mnt/data/M_logistic_regression.xlsx")

df = pd.read_excel(xlsx_in)

required = {
    "year", "region", "sex", "age_group", "bacteria", "antibiotic",
    "n_tested", "n_nonsusceptible"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# -----------------------------
# SETTINGS
# -----------------------------
pairs = [
    ("Escherichia coli", "Ceftriaxone"),
    ("Escherichia coli", "Ciprofloxacin"),
    ("Klebsiella pneumoniae", "Ceftriaxone"),
    ("Klebsiella pneumoniae", "Meropenem"),
]

sex_ref = "F"                      # Female
age_ref = "20-39"                  # Baseline
region_ref = "North Kazakhstan"    # Baseline

COLORS = {"Ceftriaxone": "tab:blue", "Ciprofloxacin": "tab:orange", "Meropenem": "tab:orange"}
sex_map = {"F": "Female", "M": "Male"}

# -----------------------------
# FILTER (safety)
# -----------------------------
df = df[df.apply(lambda r: (r["bacteria"], r["antibiotic"]) in pairs, axis=1)].copy()

# Validate baselines exist
for col, ref in [("sex", sex_ref), ("age_group", age_ref), ("region", region_ref)]:
    if ref not in set(df[col].astype(str).unique()):
        raise ValueError(f"Reference '{ref}' not found in '{col}'. Available: {sorted(df[col].astype(str).unique())}")

# -----------------------------
# POOLED
# -----------------------------
pooled = (
    df.groupby(["bacteria", "antibiotic", "region", "sex", "age_group"], as_index=False)[
        ["n_tested", "n_nonsusceptible"]
    ]
    .sum()
    .rename(columns={"n_tested": "n_tested_pooled", "n_nonsusceptible": "n_ns_pooled"})
)
pooled = pooled[pooled["n_tested_pooled"] > 0].copy()
pooled["share_ns_pooled"] = pooled["n_ns_pooled"] / pooled["n_tested_pooled"]

# Ensure categorical refs
def set_cat_with_ref(series: pd.Series, ref: str) -> pd.Categorical:
    levels = list(pd.unique(series.astype(str)))
    if ref in levels:
        levels = [ref] + [x for x in levels if x != ref]
    return pd.Categorical(series.astype(str), categories=levels, ordered=True)

pooled["sex"] = set_cat_with_ref(pooled["sex"], sex_ref)
pooled["age_group"] = set_cat_with_ref(pooled["age_group"], age_ref)
pooled["region"] = set_cat_with_ref(pooled["region"], region_ref)

# -----------------------------
# FIT MODELS -> RESULTS LONG + BASELINES
# -----------------------------
def fit_model(data_one: pd.DataFrame):
    formula = (
        f"share_ns_pooled ~ "
        f"C(region, Treatment(reference='{region_ref}')) + "
        f"C(sex, Treatment(reference='{sex_ref}')) + "
        f"C(age_group, Treatment(reference='{age_ref}'))"
    )
    return smf.glm(
        formula=formula,
        data=data_one,
        family=sm.families.Binomial(),
        freq_weights=data_one["n_tested_pooled"]
    ).fit()

rows = []
for bac, ab in pairs:
    d = pooled[(pooled["bacteria"] == bac) & (pooled["antibiotic"] == ab)].copy()
    if d.empty:
        continue

    m = fit_model(d)
    params = m.params
    conf = m.conf_int()
    pvals = m.pvalues

    res = pd.DataFrame({
        "term": params.index,
        "coef": params.values,
        "aOR": np.exp(params.values),
        "CI_low": np.exp(conf[0].values),
        "CI_high": np.exp(conf[1].values),
        "p_value": pvals.values,
    })
    res = res[res["term"] != "Intercept"].copy()
    res["bacteria"] = bac
    res["antibiotic"] = ab
    rows.append(res)

results = pd.concat(rows, ignore_index=True)

# Parse terms
def parse_term(term: str):
    if "C(sex" in term:
        lvl = term.split("[T.")[-1].rstrip("]")
        return ("sex", sex_map.get(lvl, lvl))
    if "C(age_group" in term:
        lvl = term.split("[T.")[-1].rstrip("]")
        return ("age_group", lvl)
    if "C(region" in term:
        lvl = term.split("[T.")[-1].rstrip("]")
        return ("region", lvl)
    return ("other", term)

parsed = results["term"].astype(str).apply(parse_term)
results["variable"] = parsed.apply(lambda x: x[0])
results["level"] = parsed.apply(lambda x: x[1])
results = results[results["variable"].isin(["sex", "age_group", "region"])].copy()

# Add baseline rows (OR=1.00)
baseline_rows = []
for bac, ab in pairs:
    if not ((results["bacteria"] == bac) & (results["antibiotic"] == ab)).any():
        continue
    for var, lvl in [("age_group", age_ref), ("sex", "Female"), ("region", region_ref)]:
        baseline_rows.append({
            "bacteria": bac, "antibiotic": ab, "variable": var, "level": lvl,
            "aOR": 1.0, "CI_low": np.nan, "CI_high": np.nan, "p_value": np.nan
        })

res_long = pd.concat(
    [results[["bacteria","antibiotic","variable","level","aOR","CI_low","CI_high","p_value"]],
     pd.DataFrame(baseline_rows)],
    ignore_index=True
)

# -----------------------------
# PLOT HELPERS
# -----------------------------
def set_numeric_log_ticks(ax, ticks):
    """Force numeric major ticks (without changing xlim)."""
    xmin, xmax = ax.get_xlim()
    use = [t for t in ticks if (t > 0) and (t >= xmin) and (t <= xmax)]
    if not use:
        use = [1.0]
    ax.xaxis.set_major_locator(FixedLocator(use))
    ax.xaxis.set_major_formatter(FixedFormatter([str(t).rstrip("0").rstrip(".") for t in use]))
    ax.tick_params(axis="x", which="major", labelsize=10)

def build_row_order(df_pair: pd.DataFrame):
    ages = df_pair[df_pair["variable"] == "age_group"]["level"].astype(str).unique().tolist()
    sexes = df_pair[df_pair["variable"] == "sex"]["level"].astype(str).unique().tolist()
    regs  = df_pair[df_pair["variable"] == "region"]["level"].astype(str).unique().tolist()

    def age_key(s):
        s = str(s)
        if "≥" in s or ">=" in s or s.startswith("60"):
            return (0, s)
        if s.startswith("40"):
            return (1, s)
        if s == age_ref:
            return (2, s)  # baseline
        if s.startswith("0"):
            return (3, s)
        return (9, s)

    ages_sorted = sorted(ages, key=age_key)

    sex_sorted = []
    if "Female" in sexes: sex_sorted.append("Female")
    if "Male" in sexes: sex_sorted.append("Male")
    for s in sorted(sexes):
        if s not in sex_sorted:
            sex_sorted.append(s)

    regs_sorted = []
    if region_ref in regs: regs_sorted.append(region_ref)
    regs_sorted += sorted([r for r in regs if r != region_ref])

    return ages_sorted + [""] + sex_sorted + [""] + regs_sorted

def make_forest(bacteria, antibiotics, title, out_png, tick_mode):
    sub = res_long[(res_long["bacteria"] == bacteria) & (res_long["antibiotic"].isin(antibiotics))].copy()
    if sub.empty:
        raise ValueError(f"No rows for {bacteria} / {antibiotics}")

    sub_first = sub[sub["antibiotic"] == antibiotics[0]].copy()
    rows = build_row_order(sub_first)

    frames = {ab: sub[sub["antibiotic"] == ab].set_index("level") for ab in antibiotics}

    fig, ax = plt.subplots(figsize=(9, 10))
    y = np.arange(len(rows))

    ax.axvline(1, color="grey", linewidth=2.2, zorder=0)
    ax.grid(True, which="both", axis="x", linewidth=0.4, alpha=0.35)

    offsets = np.linspace(-0.12, 0.12, num=len(antibiotics))

    # points + CI
    for off, ab in zip(offsets, antibiotics):
        tmp = frames[ab]
        xs, low, high, ys = [], [], [], []
        for i, lbl in enumerate(rows):
            if lbl == "" or lbl not in tmp.index:
                continue
            r = tmp.loc[lbl]
            aor = float(r["aOR"])
            lo = r["CI_low"]; hi = r["CI_high"]
            if pd.isna(lo) or pd.isna(hi):
                continue
            xs.append(aor)
            low.append(aor - float(lo))
            high.append(float(hi) - aor)
            ys.append(i + off)

        if xs:
            ax.errorbar(xs, ys, xerr=[low, high], fmt="o", capsize=3, linestyle="none",
                        color=COLORS.get(ab, None), label=ab, zorder=3)

    # baseline diamonds (once)
    tmp0 = frames[antibiotics[0]]
    for i, lbl in enumerate(rows):
        if lbl == "" or lbl not in tmp0.index:
            continue
        r = tmp0.loc[lbl]
        if pd.isna(r["CI_low"]) or pd.isna(r["CI_high"]):
            ax.plot(1.0, i, marker="D", markersize=6, color="black", zorder=4)

    ax.set_xscale("log")
    ax.set_yticks(y)
    ax.set_yticklabels(rows)
    ax.invert_yaxis()
    ax.set_xlabel("Adjusted odds ratio (log scale)")
    ax.set_title(title)

    # Variant A ticks (numeric on both)
    if tick_mode == "ecoli":
        set_numeric_log_ticks(ax, ticks=[0.5, 1, 2, 4])
    elif tick_mode == "kp":
        set_numeric_log_ticks(ax, ticks=[0.1, 1, 10])

    baseline_handle = Line2D([0],[0], marker="D", linestyle="none", markersize=6,
                             markerfacecolor="black", markeredgecolor="black",
                             label="Baseline (OR=1.00)")

    antibiotic_legend = ax.legend(title="Antibiotic", loc="upper left")
    ax.add_artist(antibiotic_legend)
    ax.legend(handles=[baseline_handle], loc="upper left", bbox_to_anchor=(0.0, 0.86), frameon=True)

    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {out_png}")

# -----------------------------
# MAKE 2 PLOTS
# -----------------------------
make_forest(
    bacteria="Escherichia coli",
    antibiotics=["Ceftriaxone", "Ciprofloxacin"],
    title="E. coli",
    out_png=out_dir / "forest_Ecoli_variantA.png",
    tick_mode="ecoli",
)

make_forest(
    bacteria="Klebsiella pneumoniae",
    antibiotics=["Ceftriaxone", "Meropenem"],
    title="K. pneumoniae",
    out_png=out_dir / "forest_Kpneumoniae_variantA.png",
    tick_mode="kp",
)